In [48]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [49]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = 'claude-sonnet-4-5'

The Create function is the core of making api requests , requires 3 key parameters - name of the model, max_tokens, the message. may tokemn is a ough limit and claude does not try to hit that limit its an appropriate upper bound



In [50]:
#making my first request
message = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [51]:
#ectractiog the content
message.content[0].text

'Quantum computing uses quantum bits (qubits) that exploit superposition and entanglement to process information in fundamentally different ways than classical computers, potentially solving certain complex problems exponentially faster.'

In [52]:
#claude doesnt remember the previous conversations , if u want to have a continuous conversation u need to send the earlier messages back to claude whend
#doing the API call

#Building helper Functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text
   

In [53]:
# putting it all together

# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

print(final_answer)

Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits which are either 0 or 1.


# System Prompts

In [54]:
# customize how claude responds - 
# basic structure

system_prompt = """ you are a math tutor. do not give the answer directly.guide the student through the solution"""

client.messages.create(
    model=model,
    messages=messages,
    max_tokens=1000,
    system=system_prompt
)

Message(id='msg_011CffgxqB2PLSSVPsEe8n8b', container=None, content=[TextBlock(citations=None, text='Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits which can only be 0 or 1 at any given time.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=81, output_tokens=41, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)

In [55]:
# building a flexible chat function

def chat (messages, system = None):
    params = {
        'model' : 'claude-haiku-4-5',
        'max_tokens' : 1000,
        'messages' : messages,
    }
    if system :
        params['system'] = system

    message = client.messages.create(**params)
    return message.content[0].text    

In [56]:
# without system prompt
answer = chat(messages)

# with the sys prompt

system = """you are a math tutor be patient dont give the answer upright guide the student"""

answer = chat(messages, system=system)

# Temperature
### controls how predictable ot creative the response gonna be

In [57]:
# implementing temperature in code

def chat(messages, system=None, temperature = 1.0,stop_sequences=[]):
    params = {
        'model' : 'claude-haiku-4-5',
        'max_tokens' : 1000,
        'messages' : messages,
        'temperature' : temperature,
        'stop_sequences': stop_sequences
    }

    if system:
        params['system'] = system

    message = client.messages.create(**params)
    return message.content[0].text    

In [58]:
# testing temperature effects

# low - more predictable
answer1 = chat(messages,temperature=0.0)

# hight - more creative
answer = chat(messages, temperature=1.0)

# At temperature 0.0, you might consistently get responses like "A time-traveling archaeologist must prevent ancient artifacts from being stolen." At temperature 1.0, you'll see much more variety in themes, characters, and plot elements.

# Response Streaming

In [59]:
# responses can take time to generate , making users stare at a blank screen , remedy is to give the text chunk by chunk when it is generating
# Basic Streaming Implementation

messages = []
add_user_message(messages,'write a 1 sentence description of a fake database')

stream = client.messages.create(
    model = 'claude-haiku-4-5',
    max_tokens = 1000,
    messages = messages,
    stream = True
)

for even in stream:
    print(even)


RawMessageStartEvent(message=Message(id='msg_011CffgyDP1cDjknR8foW4nv', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeDB', type='text_delta'), index=0, type

In [60]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

A relational database containing fabricated customer records, synthetic transaction histories, and randomly generated product inventories used for software testing and development purposes.

In [61]:
# Getting the final message
# need the total message for further processing

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()

# best of both worlds - real time straming for the user s and complete message object for the application logic

# Structured Data

In [62]:
# need to generate structured data - nothing else around it like codes but nothing like texts around it 
# default claude wrap the structured data around markdown for explalantion
# Solution - assistant message prefiling + stop sequences
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

# predilled assiatant emssage makes claude think that the markdown is alsready started so writes only the json contenr

In [63]:
# processing the response
import json

clean_json = json.loads(text.strip())

# Prompt Evaluation


In [64]:
# prompt engineering - writing better prompts 
# prompt evaluation - seeing how good those prompts worked
'''Why Most Engineers Fall Into Testing Traps
Options 1 and 2 are common traps that all engineers fall into, myself included. It's natural to write a prompt for a serious application and not test it thoroughly enough. We tend to underestimate how many edge cases real users will encounter.

The reality is that when you deploy a prompt to production, users will interact with it in ways you never anticipated. What seemed like a solid prompt during your limited testing can quickly break down when faced with the full variety of real-world inputs.'''

# the evaluation first approach runs the prompt through an evaluation pipeline to check it is good enough or not



"Why Most Engineers Fall Into Testing Traps\nOptions 1 and 2 are common traps that all engineers fall into, myself included. It's natural to write a prompt for a serious application and not test it thoroughly enough. We tend to underestimate how many edge cases real users will encounter.\n\nThe reality is that when you deploy a prompt to production, users will interact with it in ways you never anticipated. What seemed like a solid prompt during your limited testing can quickly break down when faced with the full variety of real-world inputs."

## A typical eval workflow

In [65]:
## step 1 draft a prompt

prompt = f"""
please answer the user's question:
{question}"""

NameError: name 'question' is not defined

### Step 02
+ create an eval dataset
contains types of questions or requests that the prompt will handle in procution - for this im gonna include 3 quesitons

+ whats 2+2?
+ how do i make oatmeal
+ how far away is the moon

### Step 03
+ feed through claude
take each questions from the dataset and merge it with the prompt template to create complete prompts and send each one to claude to get responses

`Please answer the user's question:
What's 2+2?`

### step 04 
+ feed through a grader
evaluates the quality of clauds responses by examining the original quesiotn and the answer. objective scoring is done in a scale from 1 to 10 typically 10 perfect answer5


### step 05
+ change prompt and repeat
now there is a baseline score u can run modify the prompt and run the entire process to make it improve the performance

example
´prompt = f"""
Please answer the user's question:

{question}

Answer the question with ample detail
"""´

**Prompt Scoring**
+ compare differnet prompt versions
+ use the best score propmt version
+ continue iterating to find the best one



# Generating Test Datasets

**Setting up the goal**
The prompt needs to assist users in writing three specific types of outputs
+ python code
+ JSON files
+ regular expiriences

The key requirement is that when a user requests help with a task, we return clean output in one of these formats without any extra explanations, headers, or footers.

In [67]:
# starting prompt
prompt = f"""
please provide a solution for the following task:
{task}
"""

NameError: name 'task' is not defined

In [71]:
# crreating an evaluation dataset
# generating dataset with code

def add_user_message(messages,text):
    user_message = {'role':'user','content':text}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message = {'role':'assistant','content':text}
    messages.append(assistant_message)

def chat(messages,system = None, temperature=1.0, stop_sequences=[]):
    params = {
        'model':model,
        'max_tokens' : 1000,
        'messages': messages,
        'temperature' :temperature
    }
    if system:
        params['system'] = system
    if stop_sequences:
        params['stop_sequences'] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text        

In [72]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [74]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that takes an S3 bucket ARN and extracts just the bucket name from it. For example, 'arn:aws:s3:::my-bucket-name' should return 'my-bucket-name'."}, {'task': "Create a JSON object for an AWS IAM policy that allows read-only access to a specific S3 bucket named 'company-logs', including the ability to list objects and get object contents."}, {'task': "Write a regex pattern that validates AWS EC2 instance IDs, which start with 'i-' followed by either 8 or 17 hexadecimal characters (e.g., 'i-1234abcd' or 'i-0123456789abcdef0')."}]


In [75]:
# saving the dataset

with open('dataset.json','w') as f :
    json.dump(dataset,f,indent=2)

## Running the EVAL

In [77]:
# evaluation pipeline consists of three main functions - the simplest one
# the run_prompt function

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
# the run_test_case Function
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading. logic for now hardcoded score of 10 is done
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [79]:
# the run eval function
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results


In [80]:
# Running the evaluation
with open('dataset.json','r') as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [81]:
# examining the results
print(json.dumps(results, indent = 2))

[
  {
    "output": "Looking at this task, I need to parse an S3 bucket ARN and extract just the bucket name.\n\nThe ARN format for S3 buckets is: `arn:aws:s3:::bucket-name`\n\nHere's my solution:\n\n```python\ndef extract_bucket_name(arn):\n    \"\"\"\n    Extracts the bucket name from an S3 bucket ARN.\n    \n    Args:\n        arn (str): The S3 bucket ARN (e.g., 'arn:aws:s3:::my-bucket-name')\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # S3 bucket ARN format: arn:aws:s3:::bucket-name\n    # Should have at least 6 parts (the last part is the bucket name)\n    if len(parts) < 6:\n        raise ValueError(f\"Invalid S3 bucket ARN format: {arn}\")\n    \n    # Validate the ARN structure\n    if parts[0] != 'arn' or parts[

# Model Based Grading

+ Code graders - Programmatically evaluate outputs using custom logic
+ Model graders - Use another AI model to assess the quality
+ Human graders - Have people manually review and score outputs

In [87]:
# implementing a model grader

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)



